<h1>Data Analysis</h1>

In [1]:
!pip install kagglehub

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from datetime import datetime
from sklearn.metrics import r2_score,mean_squared_error, root_mean_squared_error

In [3]:
import kagglehub

# Download latest version
#path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")

#print("Path to dataset files:", path)

c:\Users\omars\Documents\Real_Estate_Analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Download data into dataframe
data_df = pd.read_csv('datasets/usa-real-estate-dataset/versions/25/realtor-data.zip.csv',dtype={'zip_code' : str})
data_df.head(8)

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN
5,103378.0,for_sale,179000.0,4.0,3.0,0.46,1850806.0,San Sebastian,Puerto Rico,00612,2520.0,NaN
6,1205.0,for_sale,50000.0,3.0,1.0,0.20,1298094.0,Ciales,Puerto Rico,00639,2040.0,NaN
7,50739.0,for_sale,71600.0,3.0,2.0,0.08,1048466.0,Ponce,Puerto Rico,00731,1050.0,NaN


In [5]:
crime_risk_stats_df = pd.read_csv(
    "datasets/(Copy) VT/usa_pl_2024 crimerisk-statistics.csv"
)

crime_risk_stats_df.head()

,VARIABLE,DESCRIPTION,SUM,AVERAGE,MAXIMUM,MINIMUM,ZERO_RECS
0,CRMA4PERC,Personal Crime,31021389,968.298811,908062,0,442
1,CRMA4MURD,Murder,30843375,962.742298,892768,0,464
2,CRMA4RAPE,Rape,31012348,968.016606,900269,0,461
3,CRMA4ROBB,Robbery,31028107,968.508506,921872,0,492
4,CRMA4ASST,Assault,31066074,969.693604,922302,0,461


In [6]:
sq_ft_max = data_df.house_size.max()
sq_ft_min = data_df.house_size.min()

<h4>Cleaning data</h4>

In [7]:
#find null data in house size, zip code, state, city
data_df_clean = data_df.dropna(subset=['brokered_by', 'status', 'price','bed','bath','acre_lot','street','city','state','zip_code','house_size'])
data_df_clean.shape

(1354105, 12)

In [ ]:
#data_df_clean['zip_code']= data_df_clean['zip_code'].astype(str)
temp_df = pd.read_csv("datasets/(Copy) VT/usa_zi_premium_environment.csv",dtype={"ID":str})
#firstelem = data_df_clean.loc[0]
#firstelem['zip_code']

'00601'

In [11]:
#Encoding
from sklearn.preprocessing import OneHotEncoder

In [12]:
enc = OneHotEncoder(handle_unknown='ignore')
#Encode status
status = {'status' : data_df_clean.status}
statusdf = pd.DataFrame(status)
status_ohe = pd.get_dummies(statusdf, dtype = int)
#print(status_ohe)

#state
state = {'state': data_df_clean["state"]}
statedf = pd.DataFrame(state)
state_ohe = pd.get_dummies(statedf, dtype=int)
#city
data_df_clean['prev_sold_date'].fillna('1970-1-2', inplace=True)

C:\Users\omars\AppData\Local\Temp\ipykernel_16408\2126418109.py:13: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data_df_clean['prev_sold_date'].fillna('1970-1-2', inplace=True)


0            1970-1-2
1            1970-1-2
2            1970-1-2
3            1970-1-2
5            1970-1-2
              ...    
2226377    2022-03-25
2226378    2022-03-25
2226379    2022-03-24
2226380    2022-03-24
2226381    2022-03-23
Name: prev_sold_date, Length: 1354105, dtype: str

In [ ]:
#current = datetime.now()
#current.timestamp()

1773417582.867608

In [14]:
#concatenate and drop
data_df_clean = data_df_clean.drop(labels=['status','state', 'city'], axis = 'columns')
data_df_clean = pd.concat([data_df_clean,status_ohe, status_ohe], axis = 'columns')
data_df_clean.drop(labels=['prev_sold_date'],inplace=True, axis=1)
def convert_date(date_str):
    newdt = datetime.strptime(date_str, "%Y-%m-%d")
    
    print(newdt)
    return newdt.timestamp()
#Convert data to pandas datatime
#data_df_clean['prev_sold_date'] =data_df_clean['prev_sold_date'].apply(convert_date)


In [15]:
#split into X and Y
X = data_df_clean.drop(labels=['price'], axis = 'columns')
Y = data_df_clean['price']
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.33, random_state=42, shuffle=True)


In [16]:
clf = linear_model.Lasso(alpha=0.3)
#NOTE TO SELF: FIX THIS!
clf.fit(X_train,y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.3
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=1000The maximum number of iterations.",1000
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary `.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary `.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'


In [17]:
y_pred = clf.predict(X_test)
y_test = y_test.to_numpy()
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

TypeError: can't multiply sequence by non-int of type 'float'